In [169]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
month = 202511
# 用于判断是否是新品
cur_date = '2025-10-31'
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到生命周期，所以这里需要PLM的生命周期全表


### MAP信息汇总

In [170]:
Inefficient_standard_map = {
'吸油烟机':6000/12,
'灶具':6000/12,
'烤箱':1500/12,
'蒸箱':1500/12,
'微波炉':1500/12,
'蒸烤烹饪机':1500/12,
'蒸烤微烹饪机':1500/12,
'蒸微':1500/12,
'灶消烹饪机':1500/12,
'灶蒸烹饪机':1500/12,
'灶蒸烤烹饪机':1500/12,
'消毒柜':1000/12,
'热水器':900/12,
'两用炉':900/12,
'家用净水机':400/12,
'商用净水机':400/12,
'水槽洗碗机':2400/12,
'嵌入式洗碗机':2400/12,
}
productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}
productgroup_sort_map = {
    '吸油烟机': 0,
    '灶具': 1,
    '蒸烤微合计': 2,
    '灶集成': 3,
    '消毒柜': 4,
    '热水器': 5,
    '净水机': 6,
    '洗碗机': 7
}
pro_group_type_map={
    '吸油烟机': '吸油烟机',
    '灶具': '灶具',
    '烤箱': '蒸烤微合计',
    '蒸箱': '蒸烤微合计',
    '微波炉': '蒸烤微合计',
    '蒸烤烹饪机': '蒸烤微合计',
    '蒸烤微烹饪机': '蒸烤微合计',
    '蒸微': '蒸烤微合计',
    '灶消烹饪机': '灶集成',
    '灶蒸烹饪机': '灶集成',
    '灶蒸烤烹饪机': '灶集成',
    '灶烤烹饪机': '灶集成',
    '消毒柜': '消毒柜',
    '热水器': '热水器',
    '两用炉': '热水器',
    '家用净水机': '净水机',
    '商用净水机': '净水机',
    '水槽洗碗机': '洗碗机',
    '嵌入式洗碗机': '洗碗机'
}

### 读取单型号贡献报告中整理出的中间数据（处理了物流、财务、渠道、产品组、国内、核算价）(在这里剔除非三大渠道)

In [171]:
df = pd.read_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx')
df['商品编码'] = df['商品编码'].astype(str).map(lambda x: x[:13])
df['渠道'].value_counts()
df = df[df['渠道'].isin(['零售','工程','电商'])].reset_index(drop=True)
df
# print(len(df))

,商品编码,渠道,实际出库数量,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态
0,1001001500116,零售,12,1001001500116,吸油烟机,3358,40296,Z8T,国内,退市预警
1,1009000600033,零售,1,1009000600033,蒸烤烹饪机,3450,3450,ZK50-02-F1,国内,量产
2,1009000500035,零售,3,1009000500035,灶蒸烤烹饪机,5280,15840,JZT-ZK46-X2,国内,量产
3,1001001500131,零售,6,1001001500131,吸油烟机,2988,17928,02-Z6TA,国内,退市预警
4,1002003700049,零售,5,1002003700049,灶具,2550,12750,H8B,国内,量产
...,...,...,...,...,...,...,...,...,...,...
401024,1013000500000,电商,7,1013000500000,家用净水机,3640,25480,YCZ-JT1800-HR7,国内,停止销售
401025,1013000500001,零售,972,1013000500001,家用净水机,3720,3615840,YCZ-JT1800-HR7,国内,停止销售
401026,1013000500001,工程,86,1013000500001,家用净水机,3720,319920,YCZ-JT1800-HR7,国内,停止销售
401027,1013000100033,零售,61,1013000100033,家用净水机,2280,139080,YCZ-JT1800-01-M2E,国内,停止销售


### 将产品的生命周期相关信息匹配进来

In [172]:
df_product_life = pd.read_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx')
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).applymap(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life['开始销售时间'] = pd.to_datetime(df_product_life['开始销售时间'],format='mixed')
df_product_life['产品最早开始销售时间'] = df_product_life.groupby('物料号')['开始销售时间'].transform('min')
df_product_life_map_df = df_product_life[['物料号','产品状态','产品型号','产品最早开始销售时间']].drop_duplicates()


e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_5804\1706506264.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).applymap(lambda x: x[:13])


In [173]:
df1 = pd.merge(df,df_product_life_map_df,how='left',left_on='商品编码',right_on='物料号')
df1

,商品编码,渠道,实际出库数量,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态,物料号,产品状态,产品型号,产品最早开始销售时间
0,1001001500116,零售,12,1001001500116,吸油烟机,3358,40296,Z8T,国内,退市预警,1001001500116,退市预警,CXW-358-Z8T(不带罩),2022-12-10 12:00:00
1,1009000600033,零售,1,1009000600033,蒸烤烹饪机,3450,3450,ZK50-02-F1,国内,量产,1009000600033,量产,ZK50-02-F1,2025-06-09 12:00:00
2,1009000500035,零售,3,1009000500035,灶蒸烤烹饪机,5280,15840,JZT-ZK46-X2,国内,量产,1009000500035,量产,JZT-ZK46-X2,2025-04-29 12:00:00
3,1001001500131,零售,6,1001001500131,吸油烟机,2988,17928,02-Z6TA,国内,退市预警,1001001500131,退市预警,CXW-358-02-Z6TA(不带罩),2024-04-02 12:00:00
4,1002003700049,零售,5,1002003700049,灶具,2550,12750,H8B,国内,量产,1002003700049,量产,JZT-01-H8B-12T,2024-01-29 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401024,1013000500000,电商,7,1013000500000,家用净水机,3640,25480,YCZ-JT1800-HR7,国内,停止销售,1013000500000,停止销售,YCZ-JT1600-HR7,2022-06-08 12:00:00
401025,1013000500001,零售,972,1013000500001,家用净水机,3720,3615840,YCZ-JT1800-HR7,国内,停止销售,1013000500001,停止销售,YCZ-JT1800-HR7,2022-06-06 12:00:00
401026,1013000500001,工程,86,1013000500001,家用净水机,3720,319920,YCZ-JT1800-HR7,国内,停止销售,1013000500001,停止销售,YCZ-JT1800-HR7,2022-06-06 12:00:00
401027,1013000100033,零售,61,1013000100033,家用净水机,2280,139080,YCZ-JT1800-01-M2E,国内,停止销售,1013000100033,停止销售,YCZ-JT1800-01-M2E,2023-08-28 12:00:00


### 剔除新品，以及保留量产标准型号

In [174]:
# 筛选掉新品（新品：最早开始销售时间到现在未满6个月的）
df1 = df1[df1['产品最早开始销售时间']<=pd.Timestamp(cur_date)-pd.DateOffset(months=6)]
# 再在剩下的范围内选出有量产的标准型号
mass_standard = list(df1[df1['产品状态']=='量产']['标准型号'].drop_duplicates())
df1 = df1[df1['标准型号'].isin(mass_standard)].reset_index(drop=True)
df1

,商品编码,渠道,实际出库数量,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态,物料号,产品状态,产品型号,产品最早开始销售时间
0,1009000500035,零售,3,1009000500035,灶蒸烤烹饪机,5280,15840,JZT-ZK46-X2,国内,量产,1009000500035,量产,JZT-ZK46-X2,2025-04-29 12:00:00
1,1002003700049,零售,5,1002003700049,灶具,2550,12750,H8B,国内,量产,1002003700049,量产,JZT-01-H8B-12T,2024-01-29 12:00:00
2,1005000700021,零售,6,1005000700021,烤箱,3400,20400,KQD62F-02-M1A,国内,量产,1005000700021,量产,KQD62F-02-M1A,2024-04-28 12:00:00
3,1004001900003,零售,2,1004001900003,热水器,3050,6100,JSQ31-X16G2.i,国内,量产,1004001900003,量产,JSQ31-X16G2.i-FR-12T,2023-02-20 12:00:00
4,1001002000028,零售,5,1001002000028,吸油烟机,2638,13190,04-X5A,国内,量产,1001002000028,量产,CXW-358-04-X5A,2023-09-22 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299836,1013000200002,零售,3777,1013000200002,家用净水机,320,1208640,FQZ-02-BF1A,国内,量产,1013000200002,量产,FQZ-02-BF1A,2024-06-10 12:00:00
299837,1013000200002,电商,54,1013000200002,家用净水机,320,17280,FQZ-02-BF1A,国内,量产,1013000200002,量产,FQZ-02-BF1A,2024-06-10 12:00:00
299838,1013000500004,零售,841,1013000500004,家用净水机,3720,3128520,YCZ-JT2000-02-HR7,国内,量产,1013000500004,量产,YCZ-JT2000-02-HR7,2024-06-07 12:00:00
299839,1013000500005,零售,34,1013000500005,家用净水机,3640,123760,YCZ-JT2000-02-HR7,国内,量产,1013000500005,量产,YCZ-JT1700-02-HR7D,2024-06-07 12:00:00


### 对于物料编码进行分组数量汇总聚合

In [145]:
df_calu = df1.groupby('商品编码',as_index=False).agg(
                                                    产品型号=('产品型号','max'),
                                                    产品组=('产品组','max'),
                                                    产品最早开始销售时间=('产品最早开始销售时间','min'),
                                                    产品状态 = ('产品状态','max'),
                                                    标准型号=('标准型号','max'),
                                                    出库数量 = ('实际出库数量','sum'),
                                                    核算价=('核算价','sum'),                                                          
)
df_calu


,商品编码,产品型号,产品组,产品最早开始销售时间,产品状态,标准型号,出库数量,核算价
0,1001000200087,CXW-258-EA06,吸油烟机,2018-05-18 12:00:00,量产,EA06,26,504400
1,1001000500324,CXW-258-JQ36(不带罩),吸油烟机,2019-10-15 12:00:00,量产,JQ36,13718,24939324
2,1001000500372,CXW-358-JQ31A(不带罩）,吸油烟机,2021-10-29 12:00:00,量产,JQ31A,10478,20222540
3,1001000500375,CXW-358-JQ33A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,2879,5556470
4,1001000500376,CXW-358-JQ32A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,8359,16132870
...,...,...,...,...,...,...,...,...
608,1018000900007,JPCD12E-P3（升级版）,嵌入式洗碗机,2023-08-04 12:00:00,量产,JPCD12E-P3,3429,9429750
609,1018001000000,JPCD6E-03-G6,嵌入式洗碗机,2023-08-02 12:00:00,量产,JPCD6E-03-G6,27777,80553300
610,1018001000001,JPCD6E-03-G7,嵌入式洗碗机,2024-04-17 12:00:00,量产,JPCD6E-03-G7,15948,42262200
611,1018001100001,JBCD7E-03-Y1.i,嵌入式洗碗机,2023-08-22 12:00:00,停止销售,JBCD7E-03-Y1.i,3560,18512000


### 计算出每个产品型号的月均发货数

In [146]:
df_calu['各型号已售卖月份'] = ((pd.Timestamp(cur_date)-df_calu['产品最早开始销售时间']).dt.days/30).round()
df_calu['统计周期内已售卖月份'] = df_calu['各型号已售卖月份'].apply(lambda x: min(x, 12))
# 这里是新的计算方式，老的话，直接用出库数量/12月
df_calu['各型号统计周期内月平均发货量'] = df_calu['出库数量']/df_calu['统计周期内已售卖月份']
df_calu

,商品编码,产品型号,产品组,产品最早开始销售时间,产品状态,标准型号,出库数量,核算价,各型号已售卖月份,统计周期内已售卖月份,各型号统计周期内月平均发货量
0,1001000200087,CXW-258-EA06,吸油烟机,2018-05-18 12:00:00,量产,EA06,26,504400,91.0,12.0,2.166667
1,1001000500324,CXW-258-JQ36(不带罩),吸油烟机,2019-10-15 12:00:00,量产,JQ36,13718,24939324,74.0,12.0,1143.166667
2,1001000500372,CXW-358-JQ31A(不带罩）,吸油烟机,2021-10-29 12:00:00,量产,JQ31A,10478,20222540,49.0,12.0,873.166667
3,1001000500375,CXW-358-JQ33A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,2879,5556470,49.0,12.0,239.916667
4,1001000500376,CXW-358-JQ32A(不带罩）,吸油烟机,2021-10-29 12:00:00,停止销售,JQ31A,8359,16132870,49.0,12.0,696.583333
...,...,...,...,...,...,...,...,...,...,...,...
608,1018000900007,JPCD12E-P3（升级版）,嵌入式洗碗机,2023-08-04 12:00:00,量产,JPCD12E-P3,3429,9429750,27.0,12.0,285.750000
609,1018001000000,JPCD6E-03-G6,嵌入式洗碗机,2023-08-02 12:00:00,量产,JPCD6E-03-G6,27777,80553300,27.0,12.0,2314.750000
610,1018001000001,JPCD6E-03-G7,嵌入式洗碗机,2024-04-17 12:00:00,量产,JPCD6E-03-G7,15948,42262200,19.0,12.0,1329.000000
611,1018001100001,JBCD7E-03-Y1.i,嵌入式洗碗机,2023-08-22 12:00:00,停止销售,JBCD7E-03-Y1.i,3560,18512000,27.0,12.0,296.666667


### 计算出每个标准型号的总发货数，在依据其的产品组标准判断，各个各个标准型号是否是低效整机

In [147]:
df_progroup_stand_grouped = df_calu.groupby(['产品组','标准型号'],as_index=False).agg(
                                                                    标准型号月均发货数 = ('各型号统计周期内月平均发货量','sum')
)
df_progroup_stand_grouped
for index, row in df_progroup_stand_grouped.iterrows():
    if row['标准型号月均发货数'] < Inefficient_standard_map[row['产品组']]:
        df_progroup_stand_grouped.loc[index,'标准型号是否低效'] = '是'
    else:
        df_progroup_stand_grouped.loc[index,'标准型号是否低效'] = '否'
df_progroup_stand_grouped

,产品组,标准型号,标准型号月均发货数,标准型号是否低效
0,两用炉,L1GB32-X06,7.583333,是
1,两用炉,L1PB20-P01,17.000000,是
2,两用炉,L1PB26-P03T,55.916667,是
3,两用炉,L1PB26-P05.i,31.916667,是
4,吸油烟机,01-EMQ5T,139.166667,是
...,...,...,...,...
296,蒸烤烹饪机,ZK72-X20,168.083333,否
297,蒸烤烹饪机,ZK72-Y1.i,61.416667,是
298,蒸箱,SCD45-EX1.i,298.750000,否
299,蒸箱,SCD48-02-M1A,898.500000,否


In [ ]:
# df_progroup_stand_grouped.to_excel(fr"C:\Users\zhangbon\Desktop\低效明细.xlsx", index=False)

### 产品类别的低效标准型号、总标准型号数

In [148]:
df_progroup_stand_grouped['产品类别'] = df_progroup_stand_grouped['产品组'].apply(lambda x: pro_group_type_map[x])
df_progroup_stand_grouped
# low category
list_low = list(df_progroup_stand_grouped[df_progroup_stand_grouped['标准型号是否低效']=='是']['标准型号'])
# list_low

In [ ]:

df_result1 = df_progroup_stand_grouped.groupby('产品类别',as_index=False).agg(
                                                            低效标准型号数 = ('标准型号是否低效',lambda x: x[x=='是'].count()),
                                                            非低效标准型号占比 = ('标准型号是否低效',lambda x: x[x=='否'].count()),
                                                            标准型号总数 = ('标准型号是否低效','count')                                                        
).sort_values(by='产品类别',key=lambda x: x.map(productgroup_sort_map))
df_result1

,产品类别,低效标准型号数,非低效标准型号占比,标准型号总数
1,吸油烟机,25,62,87
4,灶具,12,38,50
7,蒸烤微合计,7,20,27
5,灶集成,3,10,13
3,消毒柜,5,17,22
6,热水器,15,28,43
0,净水机,1,9,10
2,洗碗机,17,32,49


### 统计各个渠道的低效产品型号数和产品型号数

In [152]:
df_result2 = pd.DataFrame()
df_result2['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    ### 零售
    count_vals1 = df1[(df1['产品组'].isin(v))&(df1['标准型号'].isin(list_low))&(df1['渠道']=='零售')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'零售低效产品型号数量'] = count_vals1
    count_vals2 = df1[(df1['产品组'].isin(v))&(df1['渠道']=='零售')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'零售产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'零售产品型号数量占比'] =  count_vals1/count_vals2

    ### 工程
    count_vals1 = df1[(df1['产品组'].isin(v))&(df1['标准型号'].isin(list_low))&(df1['渠道']=='工程')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'工程低效产品型号数量'] = count_vals1
    count_vals2 = df1[(df1['产品组'].isin(v))&(df1['渠道']=='工程')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'工程产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'工程产品型号数量占比'] =  count_vals1/count_vals2

    ### 电商
    count_vals1 = df1[(df1['产品组'].isin(v))&(df1['标准型号'].isin(list_low))&(df1['渠道']=='电商')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'电商低效产品型号数量'] = count_vals1
    count_vals2 = df1[(df1['产品组'].isin(v))&(df1['渠道']=='电商')]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'电商产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'电商产品型号数量占比'] =  count_vals1/count_vals2

    ### 全渠道
    count_vals1 = df1[(df1['产品组'].isin(v))&(df1['标准型号'].isin(list_low))]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'全渠道低效产品型号数量'] = count_vals1
    count_vals2 = df1[(df1['产品组'].isin(v))]['产品型号'].nunique()
    df_result2.loc[df_result2['产品类别'] == k,'全渠道产品型号数量'] = count_vals2
    df_result2.loc[df_result2['产品类别'] == k,'全渠道产品型号数量占比'] =  count_vals1/count_vals2
df_result2


,产品类别,零售低效产品型号数量,零售产品型号数量,零售产品型号数量占比,工程低效产品型号数量,工程产品型号数量,工程产品型号数量占比,电商低效产品型号数量,电商产品型号数量,电商产品型号数量占比,全渠道低效产品型号数量,全渠道产品型号数量,全渠道产品型号数量占比
0,吸油烟机,7.0,66.0,0.106061,16.0,65.0,0.246154,16.0,98.0,0.163265,30.0,141.0,0.212766
1,灶具,17.0,130.0,0.130769,9.0,59.0,0.152542,16.0,133.0,0.120301,23.0,204.0,0.112745
2,蒸烤微合计,3.0,21.0,0.142857,2.0,19.0,0.105263,8.0,32.0,0.250000,8.0,32.0,0.250000
3,灶集成,5.0,39.0,0.128205,1.0,16.0,0.062500,1.0,23.0,0.043478,5.0,43.0,0.116279
4,消毒柜,5.0,33.0,0.151515,3.0,24.0,0.125000,4.0,33.0,0.121212,5.0,45.0,0.111111
5,热水器,15.0,57.0,0.263158,1.0,21.0,0.047619,15.0,48.0,0.312500,18.0,72.0,0.250000
6,净水机,1.0,16.0,0.062500,1.0,10.0,0.100000,0.0,14.0,0.000000,1.0,18.0,0.055556
7,洗碗机,12.0,41.0,0.292683,7.0,26.0,0.269231,18.0,50.0,0.360000,21.0,58.0,0.362069


### 输出标准型号和渠道型号的统计分析

In [ ]:

#输出df_qudao和df_calu2，写到一个excel里面
with pd.ExcelWriter(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\低效统计分析结果.xlsx') as writer:
    df_result1.to_excel(writer, sheet_name='标准型号统计',index=False)
    df_result2.to_excel(writer, sheet_name='分渠道产品型号统计',index=False)

